In [ ]:
%pip install pandas numpy matplotlib

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from scipy.spatial.transform import Rotation as R

# === 1. Carica tutti i file CSV che iniziano per D_ ===
files = glob.glob("Misurazioni/D_*.csv")

# === 2. Unisci tutti i dati in un unico DataFrame ===
dfs = [pd.read_csv(f) for f in files]
data = pd.concat(dfs, ignore_index=True)

# Assicurati che sia ordinato nel tempo
data["_time"] = pd.to_datetime(data["_time"], utc=True, format="ISO8601")
data = data.sort_values(by="_time").reset_index(drop=True)

# === 3. Rimuovi offset iniziale (media delle prime 5 misure per ogni asse) ===
accel_offsets = data[["accel_x", "accel_y", "accel_z"]].head(5).mean()
gyro_offsets = data[["gyro_x", "gyro_y", "gyro_z"]].head(5).mean()

data["accel_x"] -= accel_offsets["accel_x"]
data["accel_y"] -= accel_offsets["accel_y"]
data["accel_z"] -= accel_offsets["accel_z"]

data["gyro_x"] -= gyro_offsets["gyro_x"]
data["gyro_y"] -= gyro_offsets["gyro_y"]
data["gyro_z"] -= gyro_offsets["gyro_z"]

gyro = data[['gyro_x', 'gyro_y', 'gyro_z']].values

print("Dimensione dati dopo unione:", data.shape)
print("Offset accelerometro rimosso:", accel_offsets)
print("Offset giroscopio rimosso:", gyro_offsets)

Dimensione dati dopo unione: (1071, 16)
Offset accelerometro rimosso: accel_x    0.133519
accel_y    0.022288
accel_z    1.033075
dtype: float64
Offset giroscopio rimosso: gyro_x    0.279774
gyro_y    0.306903
gyro_z   -0.006081
dtype: float64


In [2]:

dt = np.mean(np.diff(data['_time']).astype('timedelta64[ms]').astype(float)) / 1000.0  # secondi
rotations = [R.identity()]
for g in gyro:
    delta_rot = R.from_rotvec(g * dt)
    rotations.append(rotations[-1] * delta_rot)
rotations = rotations[1:]

print("Calcolo delle rotazioni completato.")
print("Numero di campioni:", len(rotations))

Calcolo delle rotazioni completato.
Numero di campioni: 1071


In [3]:
cube_vertices = np.array([
    [-1, -1, -1],
    [1, -1, -1],
    [1,  1, -1],
    [-1, 1, -1],
    [-1, -1, 1],
    [1, -1, 1],
    [1,  1, 1],
    [-1, 1, 1]
]) * 0.5

# --- Funzione per creare il cubo ruotato ---
def rotated_cube(rot):
    return rot.apply(cube_vertices)

# --- Creazione frame per ogni istante temporale ---
frames = []
for i, (rot, t) in enumerate(zip(rotations, data['_time'])):
    v = rotated_cube(rot)
    frame = go.Frame(
        data=[go.Mesh3d(
            x=v[:, 0], y=v[:, 1], z=v[:, 2],
            i=[0,0,0,1,1,2,2,3,4,4,4,5,5,6,6,7],
            j=[1,2,4,2,5,3,6,0,5,6,7,1,4,2,7,3],
            k=[2,4,5,3,6,0,7,1,6,7,4,4,5,2,6,7],
            color='royalblue', opacity=0.8
        )],
        name=str(t)
    )
    frames.append(frame)

In [4]:


# --- Layout e slider ---
fig = go.Figure(
    data=[go.Mesh3d(
        x=[], y=[], z=[],
        color='royalblue', opacity=0.8
    )],
    frames=frames
)

fig.update_layout(
    scene=dict(
        xaxis=dict(range=[-1, 1]),
        yaxis=dict(range=[-1, 1]),
        zaxis=dict(range=[-1, 1]),
        aspectmode='cube'
    ),
    updatemenus=[{
        "buttons": [
            {
                "args": [None, {"frame": {"duration": 50, "redraw": True}, "fromcurrent": True}],
                "label": "▶️ Play",
                "method": "animate"
            },
            {
                "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
                "label": "⏸️ Pause",
                "method": "animate"
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 30},
        "showactive": False,
        "type": "buttons",
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "steps": [
            {"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
             "label": str(f.name),
             "method": "animate"} for f in frames
        ],
        "transition": {"duration": 0},
        "x": 0.1,
        "len": 0.9
    }]
)

fig.show()
